# SpendGPT: NoteBook to Fix Bugs

This notebook alters the existing TiDB tables in place.

### BUG 1 : Null user possibility in transactions 
Every `transactions`, `budgets` and `alerts` row must belong to a valid user (`user_id` can never be NULL).

In [9]:
import os
from pathlib import Path

import certifi
import mysql.connector
from dotenv import load_dotenv

env_path = Path.cwd() / '.env'
print('Looking for .env at:', env_path, '- exists:', env_path.exists())
load_dotenv(env_path)

conn = mysql.connector.connect(
    host=os.getenv('TIDB_HOST'),
    port=int(os.getenv('TIDB_PORT', 4000)),
    user=os.getenv('TIDB_USER'),
    password=os.getenv('TIDB_PASSWORD'),
    database=os.getenv('TIDB_DATABASE', 'student_spend_db'),
    ssl_ca=certifi.where(),
    ssl_verify_cert=True,
    ssl_verify_identity=True,
)
cur = conn.cursor()
print('Connected to TiDB.')

Looking for .env at: c:\Users\sasir\Downloads\projecta\New folder\SpendGPT\database\.env - exists: True
Connected to TiDB.


## 1. Checking for existing NULL rows

`ALTER TABLE ... NOT NULL` fails if any row already has a NULL `user_id`.

In [2]:
TABLES = ['transactions', 'budgets', 'alerts']

null_counts = {}
for table in TABLES:
    cur.execute(f'SELECT COUNT(*) FROM {table} WHERE user_id IS NULL')
    null_counts[table] = cur.fetchone()[0]

print('Rows with NULL user_id:', null_counts)
if any(null_counts.values()):
    print('Delete these rows or assign them to a real user before running the next cell.')
else:
    print('No NULL user_id rows. Safe to continue.')

Rows with NULL user_id: {'transactions': 0, 'budgets': 0, 'alerts': 0}
No NULL user_id rows. Safe to continue.


## 2. Making `user_id` NOT NULL

Safe to re-run: setting an already-required column to `NOT NULL` changes nothing.

In [3]:
if any(null_counts.values()):
    raise RuntimeError('NULL user_id rows still exist. Fix them first (see the previous cell).')

for table in TABLES:
    try:
        cur.execute(f'ALTER TABLE {table} MODIFY COLUMN user_id INT NOT NULL')
        print(f'{table}: user_id is now NOT NULL')
    except mysql.connector.Error as e:
        print(f'{table}: FAILED - {e}')

transactions: user_id is now NOT NULL
budgets: user_id is now NOT NULL
alerts: user_id is now NOT NULL


## 3. Verification

Every table that has a `user_id` column should show `nullable = NO`.

In [4]:
cur.execute('''
    SELECT table_name, is_nullable
    FROM information_schema.columns
    WHERE table_schema = DATABASE() AND column_name = 'user_id'
    ORDER BY table_name
''')
for table_name, is_nullable in cur.fetchall():
    print(f'{table_name}: nullable = {is_nullable}')

accounts: nullable = NO
alerts: nullable = NO
budgets: nullable = NO
predictions: nullable = NO
transactions: nullable = NO
users: nullable = NO


## 4. Test: a budget with no user must be rejected

In [5]:
cur.execute('SELECT category_id FROM categories LIMIT 1')
category_id = cur.fetchone()[0]

try:
    cur.execute(
        'INSERT INTO budgets (user_id, category_id, amount_limit, start_date, end_date) '
        'VALUES (NULL, %s, 10, %s, %s)',
        (category_id, '2026-11-01', '2026-11-30'),
    )
    conn.commit()
    print('PROBLEM: the insert with a NULL user_id succeeded!')
except mysql.connector.Error as e:
    conn.rollback()
    print('Good - the insert was rejected:', e)

Good - the insert was rejected: 1048 (23000): Column 'user_id' cannot be null


In [6]:
cur.close()
conn.close()
print('Connection closed.')

Connection closed.


In [3]:
cur.execute("DELETE FROM receipt_items WHERE transaction_id = 150003")
cur.execute("DELETE FROM transactions WHERE transaction_id = 150003")

In [10]:
cur.execute("SELECT * FROM transactions WHERE user_id=2")